# MNIST Sparse Coding Demo\n\nThis notebook provides a small, supported MNIST sparse-coding walkthrough derived from the master's internship work completed at **IRIT (Samova team)**. It uses the maintained `SparseCoding` module in this repository instead of the legacy notebook code paths.\n

## What this notebook does\n\n- downloads MNIST from OpenML (cached by scikit-learn after the first successful download)\n- trains a compact sparse-coding dictionary on a small subset\n- visualizes learned atoms and a few input/reconstruction pairs\n

In [ ]:
from pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom sklearn.datasets import fetch_openml\n\nfrom SparseCoding import compute_cost, sparse_coding\n

In [ ]:
mnist = fetch_openml("mnist_784", version=1, as_frame=False)\nimages = mnist.data.astype(np.float32) / 255.0\nlabels = mnist.target.astype(int)\n\nsubset_size = 128\nsamples = images[:subset_size].T\nsubset_labels = labels[:subset_size]\nsamples.shape, subset_labels[:10]\n

In [ ]:
dictionary, codes, costs = sparse_coding(\n    samples,\n    k=16,\n    max_iter=6,\n    ista_max_iter=30,\n    alpha=0.03,\n    lambda_coef=0.1,\n    random_state=0,\n    return_costs=True,\n)\n\nprint("dictionary shape:", dictionary.shape)\nprint("codes shape:", codes.shape)\nprint("final cost:", compute_cost(samples, dictionary, codes, lambda_coef=0.1))\n

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(8, 8))\nfor axis, atom in zip(axes.ravel(), dictionary.T):\n    axis.imshow(atom.reshape(28, 28), cmap="gray")\n    axis.axis("off")\nfig.suptitle("Learned MNIST atoms")\nplt.tight_layout()\n

In [ ]:
reconstructions = (dictionary @ codes).T\nfig, axes = plt.subplots(3, 4, figsize=(8, 6))\nfor row in range(3):\n    axes[row, 0].imshow(images[row].reshape(28, 28), cmap="gray")\n    axes[row, 0].set_title(f"input {row}")\n    axes[row, 1].imshow(reconstructions[row].reshape(28, 28), cmap="gray")\n    axes[row, 1].set_title(f"recon {row}")\n    axes[row, 2].imshow(np.abs(images[row] - reconstructions[row]).reshape(28, 28), cmap="magma")\n    axes[row, 2].set_title(f"error {row}")\n    axes[row, 3].bar(np.arange(8), codes[:8, row])\n    axes[row, 3].set_title(f"codes {row}")\nfor axis in axes.ravel():\n    if hasattr(axis, "axis"):\n        axis.axis("off") if axis in axes[:, :3].ravel() else None\nplt.tight_layout()\n

In [ ]:
output_dir = Path("mnist_notebook_output")\noutput_dir.mkdir(exist_ok=True)\nnp.save(output_dir / "dictionary.npy", dictionary)\nnp.save(output_dir / "codes.npy", codes)\nnp.save(output_dir / "costs.npy", costs)\noutput_dir.resolve()\n